In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
raw_path  = Path("../data/raw")

nav_df = pd.read_csv(raw_path / "02_nav_history.csv")
print(nav_df.shape)
nav_df.head()

(46000, 3)


,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


In [3]:
print("Duplicates:", nav_df.duplicated().sum())
print("\nMissing values:\n", nav_df.isna().sum())
print("\nNAV <= 0:", (nav_df["nav"] <= 0).sum())

Duplicates: 0

Missing values:
 amfi_code    0
date         0
nav          0
dtype: int64

NAV <= 0: 0


In [4]:
nav_df["date"] = pd.to_datetime(nav_df["date"],)
print(nav_df.dtypes)

amfi_code             int64
date         datetime64[us]
nav                 float64
dtype: object


In [5]:
nav_df = nav_df.sort_values(['amfi_code', 'date']).reset_index(drop=True)
nav_df.head()

,amfi_code,date,nav
0,100016,2022-01-03,520.4608
1,100016,2022-01-04,515.0971
2,100016,2022-01-05,521.7239
3,100016,2022-01-06,515.7880
4,100016,2022-01-07,515.1639


In [6]:
nav_summary = nav_df.groupby("amfi_code").agg(
      start_date = ("date", "min"),
      end_date = ("date", "max"),
      rows = ("date", "count")
)

nav_summary.head()

,start_date,end_date,rows
amfi_code,,,
100016,2022-01-03,2026-05-29,1150
100025,2022-01-03,2026-05-29,1150
100033,2022-01-03,2026-05-29,1150
101206,2022-01-03,2026-05-29,1150
101207,2022-01-03,2026-05-29,1150


In [7]:
cleaned_nav = []

for amfi_code, group in nav_df.groupby("amfi_code"):
    group = group.sort_values("date")

    full_dates = pd.date_range(
        start=group["date"].min(),
        end=group["date"].max(),
        freq="D"
    )

    group = (
        group.set_index("date")
        .reindex(full_dates)
    )

    group["amfi_code"] = amfi_code
    group["nav"] = group["nav"].ffill()
    group = group.reset_index()

    group.rename(
        columns={"index": "date"},
        inplace=True
    )

    cleaned_nav.append(group)

clean_nav_df = pd.concat(cleaned_nav, ignore_index=True)

In [8]:
print(clean_nav_df.shape)
print('\nMissing values:\n', clean_nav_df["nav"].isna().sum())
clean_nav_df.head()

(64320, 3)

Missing values:
 0


,date,amfi_code,nav
0,2022-01-03,100016,520.4608
1,2022-01-04,100016,515.0971
2,2022-01-05,100016,521.7239
3,2022-01-06,100016,515.7880
4,2022-01-07,100016,515.1639


In [9]:
print('Duplicates:', clean_nav_df.duplicated().sum())
print('\nMissing values:\n', clean_nav_df.isna().sum())
print('\nNAV <= 0:', (clean_nav_df["nav"] <= 0).sum())

Duplicates: 0

Missing values:
 date         0
amfi_code    0
nav          0
dtype: int64

NAV <= 0: 0


In [10]:
processed_path = Path("../data/processed")

processed_path.mkdir(
   parents=True,
   exist_ok=True
)

clean_nav_df.to_csv(
   processed_path / "clean_nav_history.csv",
   index=False
)

print('clean_nav_history.csv saved successfully')

clean_nav_history.csv saved successfully


In [11]:
txn_df = pd.read_csv(raw_path / "08_investor_transactions.csv")

print(txn_df.shape)
txn_df.head()

(32778, 13)


,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending


In [12]:
print('Duplicates:', txn_df.duplicated().sum())

print('\nMissing values:\n', txn_df.isna().sum())
print('\nTransaction Types:\n', txn_df["transaction_type"].value_counts())
print('\nKYC Status:\n', txn_df["kyc_status"].value_counts())

print('\nAmount <= 0:', (txn_df["amount_inr"] <= 0).sum())

Duplicates: 0

Missing values:
 investor_id           0
transaction_date      0
amfi_code             0
transaction_type      0
amount_inr            0
state                 0
city                  0
city_tier             0
age_group             0
gender                0
annual_income_lakh    0
payment_mode          0
kyc_status            0
dtype: int64

Transaction Types:
 transaction_type
SIP           19716
Lumpsum        8095
Redemption     4967
Name: count, dtype: int64

KYC Status:
 kyc_status
Verified    30146
Pending      2632
Name: count, dtype: int64

Amount <= 0: 0


In [13]:
print(txn_df["kyc_status"].value_counts())

kyc_status
Verified    30146
Pending      2632
Name: count, dtype: int64


In [14]:
txn_df["transaction_date"] = pd.to_datetime(txn_df["transaction_date"],)

print(txn_df.dtypes)

investor_id                      str
transaction_date      datetime64[us]
amfi_code                      int64
transaction_type                 str
amount_inr                     int64
state                            str
city                             str
city_tier                        str
age_group                        str
gender                           str
annual_income_lakh           float64
payment_mode                     str
kyc_status                       str
dtype: object


In [15]:
print(txn_df["transaction_date"].min(),)
print(txn_df["transaction_date"].max(),)

2024-01-01 00:00:00
2025-05-30 00:00:00


In [16]:
txn_df.to_csv(
   processed_path / "clean_investor_transactions.csv",
   index=False
)

print('clean_investor_transactions.csv saved successfully')

clean_investor_transactions.csv saved successfully


In [17]:
perf_df = pd.read_csv(raw_path / "07_scheme_performance.csv")

print(perf_df.shape)
perf_df.head()

(40, 19)


,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low


In [18]:
print('Duplicates:', perf_df.duplicated().sum())
print('\nMissing values:\n', perf_df.isna().sum())

Duplicates: 0

Missing values:
 amfi_code             0
scheme_name           0
fund_house            0
category              0
plan                  0
return_1yr_pct        0
return_3yr_pct        0
return_5yr_pct        0
benchmark_3yr_pct     0
alpha                 0
beta                  0
sharpe_ratio          0
sortino_ratio         0
std_dev_ann_pct       0
max_drawdown_pct      0
aum_crore             0
expense_ratio_pct     0
morningstar_rating    0
risk_grade            0
dtype: int64


In [19]:
return_cols = [
   "return_1yr_pct",
   "return_3yr_pct",
   "return_5yr_pct",
   "benchmark_3yr_pct"
]

print(perf_df[return_cols].dtypes)

return_1yr_pct       float64
return_3yr_pct       float64
return_5yr_pct       float64
benchmark_3yr_pct    float64
dtype: object


In [20]:
print('Negative sharpe ratios:', (perf_df["sharpe_ratio"] < 0).sum())
print('\nMinimum expense ratio:', perf_df["expense_ratio_pct"].min())
print('maximum expense ratio:', perf_df["expense_ratio_pct"].max())

Negative sharpe ratios: 0

Minimum expense ratio: 0.55
maximum expense ratio: 1.64


In [21]:
print('SCHEME PERFORMANCE VALIDATION')
print('\nAll return columns are numeric.')
print('\nNegative sharpe rations:', (perf_df["sharpe_ratio"] < 0).sum())
print('Expense ratio range:')
print('Min:', perf_df["expense_ratio_pct"].min())
print('Max:', perf_df["expense_ratio_pct"].max())

print('\nValidation passed.')

SCHEME PERFORMANCE VALIDATION

All return columns are numeric.

Negative sharpe rations: 0
Expense ratio range:
Min: 0.55
Max: 1.64

Validation passed.


In [22]:
perf_df.to_csv(
   processed_path / "clean_scheme_performance.csv",
   index=False
)
print('clean_scheme_performance.csv saved successfully')

clean_scheme_performance.csv saved successfully


In [23]:
from sqlalchemy import create_engine
from pathlib import Path

In [24]:
db_path = Path('../data/db')
db_path.mkdir(parents=True, exist_ok=True)

engine = create_engine('sqlite:///../data/db/bluestock_mf.db')
print('Databse connection created')

Databse connection created


In [25]:
schema_file = Path("../sql/schema.sql")

with open(schema_file, "r") as f:
    schema_sql = f.read()

print('Schema loaded')

Schema loaded


In [26]:
from sqlalchemy import text

with engine.begin() as conn:
    for statement in schema_sql.split(";"):
        statement = statement.strip()

        if statement:
            conn.execute(text(statement))

print("Schema recreated successfully")

Schema recreated successfully


In [27]:
with engine.connect() as conn:
   tables = conn.execute(text("SELECT name FROM sqlite_master WHERE type='table';"))
   for table in tables:
      print(table[0])

dim_fund
dim_date
fact_nav
sqlite_sequence
fact_transactions
fact_performance
fact_aum


In [28]:
with engine.connect() as conn:
   result = conn.execute(
      text("""
         SELECT name
         FROM sqlite_master
         WHERE type='table'
         AND name != 'sqlite_sequence'
         """)
   )

   tables = [row[0] for row in result]
   print('Number of project tables:', len(tables))
   print(tables)

Number of project tables: 6
['dim_fund', 'dim_date', 'fact_nav', 'fact_transactions', 'fact_performance', 'fact_aum']


In [29]:
fund_df = pd.read_csv(
   raw_path / "01_fund_master.csv"
)

print(fund_df.shape)
fund_df.head()

(40, 15)


,amfi_code,fund_house,scheme_name,category,sub_category,plan,launch_date,benchmark,expense_ratio_pct,exit_load_pct,min_sip_amount,min_lumpsum_amount,fund_manager,risk_category,sebi_category_code
0,119551,SBI Mutual Fund,SBI Bluechip Fund - Regular Plan - Growth,Equity,Large Cap,Regular,2006-02-14,NIFTY 100 TRI,1.54,1.0,500,1000,Sohini Andani,Moderate,EC01
1,119552,SBI Mutual Fund,SBI Bluechip Fund - Direct Plan - Growth,Equity,Large Cap,Direct,2013-01-01,NIFTY 100 TRI,0.66,1.0,500,1000,Sohini Andani,Moderate,EC01
2,119598,SBI Mutual Fund,SBI Small Cap Fund - Regular Plan - Growth,Equity,Small Cap,Regular,2009-09-09,BSE 250 SmallCap TRI,1.43,1.0,500,1000,R. Srinivasan,Very High,EC03
3,119599,SBI Mutual Fund,SBI Small Cap Fund - Direct Plan - Growth,Equity,Small Cap,Direct,2013-01-01,BSE 250 SmallCap TRI,0.72,1.0,500,1000,R. Srinivasan,Very High,EC03
4,119120,SBI Mutual Fund,SBI Magnum Gilt Fund - Regular Plan - Growth,Debt,Gilt,Regular,2000-12-30,CRISIL Dynamic Gilt Index,0.77,0.0,500,1000,Dinesh Ahuja,Low,DC02


In [30]:
fund_df.to_sql(
   "dim_fund",
   engine,
   if_exists='append',
   index=False
)

print('dim_fund loaded successfully')

dim_fund loaded successfully


In [31]:
with engine.connect() as conn:
   result = conn.execute(
      text("SELECT COUNT(*) FROM dim_fund")
   )

   print('Rows in dim_fund:', result.scalar())

Rows in dim_fund: 40


In [32]:
clean_nav_db = pd.read_csv(
   processed_path / "clean_nav_history.csv"
)

print(clean_nav_db.shape)
clean_nav_db.head()

(64320, 3)


,date,amfi_code,nav
0,2022-01-03,100016,520.4608
1,2022-01-04,100016,515.0971
2,2022-01-05,100016,521.7239
3,2022-01-06,100016,515.7880
4,2022-01-07,100016,515.1639


In [33]:
clean_nav_db = clean_nav_db.rename(
   columns={"date": "nav_date"}
)

clean_nav_db.head()

,nav_date,amfi_code,nav
0,2022-01-03,100016,520.4608
1,2022-01-04,100016,515.0971
2,2022-01-05,100016,521.7239
3,2022-01-06,100016,515.7880
4,2022-01-07,100016,515.1639


In [34]:
clean_nav_db.to_sql(
   "fact_nav",
   engine,
   if_exists='append',
   index=False
)

print('fact_nav loaded successfully')

fact_nav loaded successfully


In [35]:
with engine.connect() as conn:
   result = conn.execute(
      text('SELECT COUNT(*) FROM fact_nav')
   )

   print('Rows in fact_nav:', result.scalar())

Rows in fact_nav: 64320


In [36]:
txn_db = pd.read_csv(
   processed_path / "clean_investor_transactions.csv"
)

print(txn_db.shape)
txn_db.head()

(32778, 13)


,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending


In [37]:
txn_db.to_sql(
   "fact_transactions",
   engine,
   if_exists='append',
   index=False
)

print('fact_transactions loaded successfully')

fact_transactions loaded successfully


In [38]:
with engine.connect() as conn:
   result = conn.execute(
      text('SELECT COUNT(*) FROM fact_transactions')
   )

   print('Rows in fact_transactions:', result.scalar())

Rows in fact_transactions: 32778


In [39]:
perf_db = pd.read_csv(
   processed_path / "clean_scheme_performance.csv"
)

print(perf_db.shape)
perf_db.head()

(40, 19)


,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low


In [40]:
perf_db = perf_db[
   [
      "amfi_code",
      "return_1yr_pct",
      "return_3yr_pct",
      "return_5yr_pct",
      "benchmark_3yr_pct",
      "alpha",
      "beta",
      "sharpe_ratio",
      "sortino_ratio",
      "std_dev_ann_pct",
      "max_drawdown_pct",
      "aum_crore",
      "expense_ratio_pct",
      "morningstar_rating",
      "risk_grade"
   ]
]

print(perf_db.shape)

(40, 15)


In [41]:
perf_db.to_sql(
   'fact_performance',
   engine,
   if_exists='append',
   index=False
)

print('fact_performance loaded successfully')

fact_performance loaded successfully


In [42]:
with engine.connect() as conn:
    result = conn.execute(
        text("SELECT COUNT(*) FROM fact_performance")
    )

    print("Rows in fact_performance:", result.scalar())

Rows in fact_performance: 40


In [43]:
aum_df = pd.read_csv(
   raw_path / "03_aum_by_fund_house.csv"
)

print(aum_df.shape)
aum_df.head()

(90, 5)


,date,fund_house,aum_lakh_crore,aum_crore,num_schemes
0,2022-03-31,SBI Mutual Fund,6.05,605000,186
1,2022-03-31,ICICI Prudential MF,4.65,465000,216
2,2022-03-31,HDFC Mutual Fund,4.35,435000,195
3,2022-03-31,Nippon India MF,2.70,270000,177
4,2022-03-31,Kotak Mahindra MF,2.70,270000,168


In [44]:
aum_df['date'] = pd.to_datetime(
   aum_df['date']
)

print(aum_df.dtypes)

date              datetime64[us]
fund_house                   str
aum_lakh_crore           float64
aum_crore                  int64
num_schemes                int64
dtype: object


In [45]:
aum_df.to_sql(
   'fact_aum',
   engine,
   if_exists='append',
   index=False
)

print('fact_sum loaded successfully')

fact_sum loaded successfully


In [46]:
with engine.connect() as conn:
   result = conn.execute(
      text('SELECT COUNT (*) FROM fact_aum')
   )

   print('Row in fact_aum', result.scalar())

Row in fact_aum 90


In [47]:
date_range = pd.date_range(
   start='2022-01-03',
   end='2026-05-29',
   freq='D'
)

dim_date = pd.DataFrame({
   'full_date': date_range
})

print(dim_date.shape)
dim_date.head()

(1608, 1)


,full_date
0,2022-01-03
1,2022-01-04
2,2022-01-05
3,2022-01-06
4,2022-01-07


In [48]:
dim_date['date_id'] = (
   dim_date['full_date']
   .dt.strftime('%Y%m%d')
   .astype(int)
)

dim_date['year'] = dim_date['full_date'].dt.year
dim_date['month'] = dim_date['full_date'].dt.month
dim_date['quarter'] = dim_date['full_date'].dt.quarter

dim_date['day_of_week'] = (
   dim_date['full_date'].dt.dayofweek
)

dim_date['is_weekday'] = (
   dim_date['day_of_week'] < 5
).astype(int)

dim_date.head()

,full_date,date_id,year,month,quarter,day_of_week,is_weekday
0,2022-01-03,20220103,2022,1,1,0,1
1,2022-01-04,20220104,2022,1,1,1,1
2,2022-01-05,20220105,2022,1,1,2,1
3,2022-01-06,20220106,2022,1,1,3,1
4,2022-01-07,20220107,2022,1,1,4,1


In [49]:
dim_date = dim_date[
   [
      'date_id',
      'full_date',
      'year',
      'month',
      'quarter',
      'day_of_week',
      'is_weekday'
   ]
]

print(dim_date.shape)
dim_date.head()

(1608, 7)


,date_id,full_date,year,month,quarter,day_of_week,is_weekday
0,20220103,2022-01-03,2022,1,1,0,1
1,20220104,2022-01-04,2022,1,1,1,1
2,20220105,2022-01-05,2022,1,1,2,1
3,20220106,2022-01-06,2022,1,1,3,1
4,20220107,2022-01-07,2022,1,1,4,1


In [50]:
print(dim_date['full_date'].min())
print(dim_date['full_date'].max())

2022-01-03 00:00:00
2026-05-29 00:00:00


In [51]:
dim_date.to_sql(
   'dim_date',
   engine,
   if_exists='append',
   index=False
)

print('dim_date loaded successfully')

dim_date loaded successfully


In [52]:
with engine.connect() as conn:
   result = conn.execute(
      text('SELECT COUNT(*) FROM dim_date')
   )

   print('Rows in dim_date:', result.scalar())

Rows in dim_date: 1608


In [53]:
tables = [
    "dim_fund",
    "dim_date",
    "fact_nav",
    "fact_transactions",
    "fact_performance",
    "fact_aum"
]

with engine.connect() as conn:
    for table in tables:
        result = conn.execute(
            text(f"SELECT COUNT(*) FROM {table}")
        )
        print(f"{table}: {result.scalar()}")

dim_fund: 40
dim_date: 1608
fact_nav: 64320
fact_transactions: 32778
fact_performance: 40
fact_aum: 90


In [54]:
from sqlalchemy import text

with engine.connect() as conn:

    result = conn.execute(text("""
        SELECT
            transaction_type,
            ROUND(AVG(amount_inr), 2) AS avg_amount
        FROM fact_transactions
        GROUP BY transaction_type
    """))

    for row in result:
        print(row)

('Lumpsum', 254456.02)
('Redemption', 250558.79)
('SIP', 11018.13)


In [55]:
with engine.connect() as conn:

    result = conn.execute(text("""
        SELECT
            risk_category,
            COUNT(*) AS fund_count
        FROM dim_fund
        GROUP BY risk_category
        ORDER BY fund_count DESC
    """))

    for row in result:
        print(row)

('Moderate', 16)
('High', 8)
('Very High', 6)
('Low', 6)
('Moderately High', 4)
